# 02 — Preprocessing pipeline walk-through

Step through the same pipeline that `scripts/build_dataset.py` runs, but for a single TIC, with intermediate plots so you can see what each step does.

**Pipeline:**
1. Download → stitched multi-sector light curve.
2. Sigma clip + remove NaNs.
3. Flatten with Savitzky-Golay (removes slow stellar variability).
4. Phase-fold on (period, t0).
5. Bin to fixed-length global + local arrays (Shallue & Vanderburg 2018 convention).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from exoplanet_hunter.data.download import LightCurveDownloader
from exoplanet_hunter.preprocess import build_views, clean_lightcurve, flatten_lightcurve
from exoplanet_hunter.utils import ProjectPaths
from omegaconf import OmegaConf

import lightkurve as lk

# Set up paths the way Hydra would.
cfg = OmegaConf.create({"paths": {
    "root":           "..",
    "data_raw":       "../data/raw",
    "data_interim":   "../data/interim",
    "data_processed": "../data/processed",
    "data_labels":    "../data/labels",
    "models":         "../models",
    "results":        "../results",
}})
paths = ProjectPaths.from_cfg(cfg)

TIC = 150428135

In [ ]:
dl = LightCurveDownloader(paths.data_raw, author="SPOC", cadence=120)
result = dl.download_one(TIC)
lc = lk.read(str(result.path))
print(f"sectors: {result.n_sectors}, points: {result.n_points}")
lc.plot();

In [ ]:
lc_clean = clean_lightcurve(lc, sigma_clip=5.0, min_points=100)
lc_flat  = flatten_lightcurve(lc_clean, window_length=301, polyorder=2)

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
axes[0].plot(lc_clean.time.value, lc_clean.flux.value, '.', ms=1); axes[0].set_title("cleaned")
axes[1].plot(lc_flat.time.value,  lc_flat.flux.value,  '.', ms=1); axes[1].set_title("flattened")
plt.tight_layout();

In [ ]:
views = build_views(lc_flat, period=9.977, t0=1493.62, duration=1.6/24)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(views.global_view); axes[0].set_title("global view")
axes[1].plot(views.local_view);  axes[1].set_title("local view")
plt.tight_layout();

**Try yourself:**
- Vary `window_length` for the flatten step. What happens at `window_length=51`? (hint: shorter than transit → transit gets removed)
- Compare global view binned at 2001 vs 501 — lower resolution loses the secondary-eclipse window.
- Plot the local view with `local_durations=1.0` vs `3.0` — wider gives more out-of-transit context.